# Mini-Project 1: Weather Forecast with Python Web APIs

Data sources: Open-Meteo (geocoding and forecast) and MET Norway Locationforecast 2.0.

In [14]:
# --- CONFIGURATION: the only cell to edit ---
import os

LOCATION = "MUKONO"  # city to forecast; change if you like
COUNTRY_CODE = "UG"  # ISO country code to narrow the search; None = worldwide
LOCATION_CHOICE = 0  # number of the place to use, from the candidate list below

PLACEHOLDER_EMAIL = "YOUR_UNIVERSITY_EMAIL"
CONTACT_EMAIL = os.environ.get("WEATHER_CONTACT_EMAIL", PLACEHOLDER_EMAIL)
USER_AGENT = f"oop-python-assignment2/1.0 ({CONTACT_EMAIL})"

if CONTACT_EMAIL == PLACEHOLDER_EMAIL:
    print("WARNING: set WEATHER_CONTACT_EMAIL before calling the APIs.")
print("Location:", LOCATION)
print("Contact email configured:", CONTACT_EMAIL != PLACEHOLDER_EMAIL)

Location: MUKONO
Contact email configured: True


In [2]:
import json
import urllib.parse
import urllib.request

GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"

query = urllib.parse.urlencode(
    {"name": LOCATION, "count": 5, "language": "en", "format": "json"}
)
url = f"{GEOCODING_URL}?{query}"
req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})

with urllib.request.urlopen(req, timeout=10) as resp:
    status_urllib = resp.status
    body = resp.read().decode("utf-8")

data_urllib = json.loads(body)
print("URL:", url)
print("Status:", status_urllib)
for place in data_urllib.get("results", []):
    print(place["name"], "|", place.get("admin1"), "|",
          place.get("country"), "|", place["latitude"], place["longitude"])

URL: https://geocoding-api.open-meteo.com/v1/search?name=MUKONO&count=5&language=en&format=json
Status: 200
Mukono | Central Region | Uganda | 0.35333 32.75528
Mukono | Northern Province | Rwanda | -1.4551 30.0149
Mokono | Kasai | Democratic Republic of Congo | -7.28013 21.62154
Mukono | Central Region | Uganda | 0.27276 32.46589
Mukono | Western Province | Zambia | -14.6 24.25


In [3]:
import requests

resp_requests = requests.get(
    GEOCODING_URL,
    params={"name": LOCATION, "count": 5, "language": "en", "format": "json"},
    headers={"User-Agent": USER_AGENT},
    timeout=10,
)
data_requests = resp_requests.json()

print("URL built by requests:", resp_requests.url)
print("Status:", resp_requests.status_code)
print("Number of results:", len(data_requests.get("results", [])))
print("Same results as urllib:",
      data_requests.get("results") == data_urllib.get("results"))

URL built by requests: https://geocoding-api.open-meteo.com/v1/search?name=MUKONO&count=5&language=en&format=json
Status: 200
Number of results: 5
Same results as urllib: True


##  urllib vs requests

| Aspect | urllib (standard library) | requests |
|---|---|---|
| Code length | 6 steps. urlencode, then Request, urlopen, read, decode and json.loads | 2 steps. requests.get(...) and .json() |
| Error handling | raises HTTPError automatically for 4xx and 5xx responses, and URLError for network failures. | does not raise for 4xx or 5xx, so you must call raise_for_status(). Network failures raise ConnectionError or Timeout, both under RequestException |
| Header control | you must build a Request object to add headers. | pass a headers= dictionary directly. | 
| Encoding | returns raw bytes. The code decoded them manually, assuming UTF-8. | reads charset=utf-8 from the Content-Type header and decodes automatically |

**Conclusion:**  Both methods gave the same results for Mukono. I'll use `requests` for the rest of the project because it's shorter and handles headers, parameters and encoding for me. The catch is that it doesn't flag 4xx/5xx errors on its own, so I'll need `raise_for_status()`. `urllib` is still handy when you can't install extra packages.

In [5]:
print("Status code:", resp_requests.status_code)
print("Content-Type:", resp_requests.headers.get("Content-Type"))
print("Response time (s):", round(resp_requests.elapsed.total_seconds(), 3))

Status code: 200
Content-Type: application/json; charset=utf-8
Response time (s): 0.74


## Part B: Forecast retrieval and display

In [6]:
%load_ext autoreload
%autoreload 2

import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.weather import parse_geocoding_results

In [7]:
candidates = parse_geocoding_results(data_requests)
for number, place in enumerate(candidates):
    print(number, place)

print("No-results case:", parse_geocoding_results({}))


0 {'name': 'Mukono', 'admin1': 'Central Region', 'country': 'Uganda', 'latitude': 0.35333, 'longitude': 32.75528, 'timezone': 'Africa/Kampala'}
1 {'name': 'Mukono', 'admin1': 'Northern Province', 'country': 'Rwanda', 'latitude': -1.4551, 'longitude': 30.0149, 'timezone': 'Africa/Kigali'}
2 {'name': 'Mokono', 'admin1': 'Kasai', 'country': 'Democratic Republic of Congo', 'latitude': -7.28013, 'longitude': 21.62154, 'timezone': 'Africa/Lubumbashi'}
3 {'name': 'Mukono', 'admin1': 'Central Region', 'country': 'Uganda', 'latitude': 0.27276, 'longitude': 32.46589, 'timezone': 'Africa/Kampala'}
4 {'name': 'Mukono', 'admin1': 'Western Province', 'country': 'Zambia', 'latitude': -14.6, 'longitude': 24.25, 'timezone': 'Africa/Lusaka'}
No-results case: []


In [12]:
from src.weather import geocode

candidates = geocode(LOCATION, USER_AGENT, country_code=COUNTRY_CODE)
for number, place in enumerate(candidates):
    print(number, place["name"], "|", place["admin1"], "|",
          place["country"], "| population:", place["population"])

0 Mukono | Central Region | Uganda | population: 191300
1 Mukono | Central Region | Uganda | population: None
2 Mukono Landing | Central Region | Uganda | population: None


In [15]:
from src.weather import choose_location

location = choose_location(candidates, LOCATION_CHOICE)
print(f"Using: {location['name']}, {location['admin1']}, {location['country']} "
      f"({location['latitude']}, {location['longitude']}), "
      f"timezone {location['timezone']}")

Using: Mukono, Central Region, Uganda (0.35333, 32.75528), timezone Africa/Kampala
